# Chapter 6 Student Colab Notebook: Linear Transformations

**Course:** MATH 3260 - Linear Algebra  
**Sections:** 6.1-6.5  
**Audience:** Student version

This notebook is the complete interactive teaching companion for Chapter 6. It combines theory, original worked examples, complete calculations, geometric interpretation, applications, Python visualizations, interactive demonstrations, section worksheets, and complete instructor solutions.

> **How to use this student notebook**
>
> Run the cells from top to bottom in Google Colab. Read the complete theory and worked examples, explore the visualizations, and complete each section worksheet. Complete instructor solutions are not included. Display mathematics uses `$$...$$` for reliable Colab rendering.

## Chapter map

1. **Section 6.1:** Introduction to Linear Transformations  
2. **Section 6.2:** Kernel and Range  
3. **Section 6.3:** Matrices for Linear Transformations  
4. **Section 6.4:** Transition Matrices and Similarity  
5. **Section 6.5:** Applications of Linear Transformations  

Each section ends with a terminology table, a worksheet, and complete instructor solutions.

In [ ]:
# Core imports used throughout Chapter 6
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from IPython.display import display, Math, Markdown
from mpl_toolkits.mplot3d import Axes3D

try:
    from ipywidgets import interact, FloatSlider, Dropdown
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

sp.init_printing()

def show_matrix(M, label=None):
    M = sp.Matrix(M)
    if label:
        display(Math(rf"{label}={sp.latex(M)}"))
    else:
        display(M)
    return M

def show_vector(v, label=None):
    v = sp.Matrix(v)
    if label:
        display(Math(rf"{label}={sp.latex(v)}"))
    else:
        display(v)
    return v

def show_exact(expr, label):
    expr = sp.simplify(expr)
    display(Math(rf"{label}={sp.latex(expr)}"))
    return expr

def plot_vectors_2d(vectors, labels, title='Vectors in the plane'):
    fig, ax = plt.subplots(figsize=(6.4,5.4))
    maxval = max(max(abs(float(x)) for x in v) for v in vectors) + 1
    for v,label in zip(vectors,labels):
        ax.quiver(0,0,float(v[0]),float(v[1]),angles='xy',scale_units='xy',scale=1)
        ax.text(float(v[0]),float(v[1]),f'  {label}')
    ax.axhline(0,linewidth=.8); ax.axvline(0,linewidth=.8)
    ax.set_xlim(-maxval,maxval); ax.set_ylim(-maxval,maxval)
    ax.set_aspect('equal'); ax.grid(alpha=.3); ax.set_title(title)
    plt.show()

def transform_polygon(points, A):
    pts=np.asarray(points,dtype=float)
    M=np.asarray(A,dtype=float)
    return (M@pts.T).T

def plot_polygon_transform(points, A, title='Linear transformation of a figure'):
    pts=np.asarray(points,dtype=float)
    trans=transform_polygon(pts,A)
    fig,ax=plt.subplots(figsize=(7,5.5))
    ax.plot(pts[:,0],pts[:,1],marker='o',label='original')
    ax.plot(trans[:,0],trans[:,1],marker='o',label='transformed')
    ax.axhline(0,linewidth=.8); ax.axvline(0,linewidth=.8)
    allpts=np.vstack([pts,trans]); pad=1
    ax.set_xlim(allpts[:,0].min()-pad,allpts[:,0].max()+pad)
    ax.set_ylim(allpts[:,1].min()-pad,allpts[:,1].max()+pad)
    ax.set_aspect('equal'); ax.grid(alpha=.3); ax.legend(); ax.set_title(title)
    plt.show()

def rotation2(theta):
    return np.array([[np.cos(theta),-np.sin(theta)],[np.sin(theta),np.cos(theta)]])

def rotation3_z(theta):
    return np.array([[np.cos(theta),-np.sin(theta),0],[np.sin(theta),np.cos(theta),0],[0,0,1]])

> **Original-example policy**
>
> All worked examples and numerical values are original. Themes are used selectively from the San Antonio Spurs, hiking, badminton, driving, travel, Backstreet Boys, and R&B music. Any routes, listening profiles, scores, or geometric models are fictional teaching contexts unless explicitly identified otherwise.

# Section 6.1 - Introduction to Linear Transformations

## Learning objectives

Students should be able to:

- identify the domain, codomain, image, preimage, and range of a transformation;
- test whether a mapping is linear;
- use the structural properties of linear transformations;
- determine a transformation from its values on a basis;
- recognize matrix-defined transformations and calculus operators as linear.

> **Function terminology**
>
> A transformation $T:V\to W$ assigns each vector in the domain $V$ exactly one vector in the codomain $W$. If $T(\mathbf v)=\mathbf w$, then $\mathbf w$ is the image of $\mathbf v$, and $\mathbf v$ is a preimage of $\mathbf w$.

## A travel-profile transformation

Define

$$
T:\mathbb R^2\to\mathbb R^2,
\qquad
T(x,y)=(2x-y,\,x+3y).
$$

For the fictional travel profile $(3,1)$,

$$
T(3,1)=(2(3)-1,\,3+3(1))=(5,6).
$$

To find the preimage of $(5,6)$, solve

$$
\begin{aligned}
2x-y&=5,\\
x+3y&=6.
\end{aligned}
$$

From the first equation, $y=2x-5$. Substitution gives

$$
x+3(2x-5)=6
\quad\Longrightarrow\quad
7x=21
\quad\Longrightarrow\quad
x=3,
$$

and therefore $y=1$. The preimage is the single vector $(3,1)$.

In [ ]:
x,y=sp.symbols('x y')
sol=sp.solve([sp.Eq(2*x-y,5),sp.Eq(x+3*y,6)],[x,y],dict=True)
print(sol)
plot_vectors_2d([[3,1],[5,6]],['input (3,1)','image (5,6)'],'A vector and its image')

> **Definition: linear transformation**
>
> A mapping $T:V\to W$ is linear when $T(\mathbf u+\mathbf v)=T(\mathbf u)+T(\mathbf v)$ and $T(c\mathbf u)=cT(\mathbf u)$ for all vectors and all scalars.

## Verifying linearity

Let

$$
T(x,y)=(2x-y,\,x+3y).
$$

For $
\mathbf u=(u_1,u_2)$ and $
\mathbf v=(v_1,v_2)$,

$$
\begin{aligned}
T(\mathbf u+\mathbf v)
&=T(u_1+v_1,u_2+v_2)\\
&=(2u_1+2v_1-u_2-v_2,\,u_1+v_1+3u_2+3v_2)\\
&=(2u_1-u_2,\,u_1+3u_2)+(2v_1-v_2,\,v_1+3v_2)\\
&=T(\mathbf u)+T(\mathbf v).
\end{aligned}
$$

Also,

$$
T(c\mathbf u)
=T(cu_1,cu_2)
=(2cu_1-cu_2,\,cu_1+3cu_2)
=cT(\mathbf u).
$$

Therefore, $T$ is linear.

> **Fast nonlinearity test**
>
> Every linear transformation satisfies $T(\mathbf 0)=\mathbf 0$. Therefore, any mapping with $T(\mathbf 0)\neq\mathbf 0$ is immediately non-linear.

For example,

$$
F(x,y)=(x+1,y)
$$

is not linear because

$$
F(0,0)=(1,0)\neq(0,0).
$$

## A transformation determined by basis images

Suppose a linear transformation $T:\mathbb R^3\to\mathbb R^2$ satisfies

$$
T(\mathbf e_1)=(9,1),
\qquad
T(\mathbf e_2)=(20,-3),
\qquad
T(\mathbf e_3)=(21,4).
$$

Because

$$
(2,-1,3)=2\mathbf e_1-\mathbf e_2+3\mathbf e_3,
$$

linearity gives

$$
\begin{aligned}
T(2,-1,3)
&=2T(\mathbf e_1)-T(\mathbf e_2)+3T(\mathbf e_3)\\
&=2(9,1)-(20,-3)+3(21,4)\\
&=(61,17).
\end{aligned}
$$

In [ ]:
T1=sp.Matrix([9,1]); T2=sp.Matrix([20,-3]); T3=sp.Matrix([21,4])
result=2*T1-T2+3*T3
show_vector(result,r'T(2,-1,3)')

## Matrix-defined transformations

Every $m\times n$ matrix $A$ defines a linear transformation

$$
T:\mathbb R^n\to\mathbb R^m,
\qquad
T(\mathbf x)=A\mathbf x.
$$

For

$$
A=
\begin{bmatrix}
1&2\\
3&-1\\
0&4
\end{bmatrix},
\qquad
\mathbf v=
\begin{bmatrix}
2\\-1
\end{bmatrix},
$$

we obtain

$$
T(\mathbf v)
=A\mathbf v
=
\begin{bmatrix}
1(2)+2(-1)\\
3(2)-(-1)\\
4(-1)
\end{bmatrix}
=
\begin{bmatrix}
0\\7\\-4
\end{bmatrix}.
$$

In [ ]:
A=sp.Matrix([[1,2],[3,-1],[0,4]])
v=sp.Matrix([2,-1])
show_matrix(A,'A'); show_vector(A*v,r'T(\mathbf v)')

## Calculus operators

The differential operator

$$
D:P_3\to P_2,
\qquad
D(p)=p'
$$

is linear because differentiation preserves sums and scalar multiples.

For

$$
p(x)=4-3x+2x^2+x^3,
$$

$$
D(p)=-3+4x+3x^2.
$$

The definite-integral functional

$$
L(p)=\int_0^1p(x)\,dx
$$

is also linear. For $p(x)=1+2x+x^2$,

$$
L(p)=1+1+\frac13=\frac73.
$$

In [ ]:
x=sp.symbols('x')
p=4-3*x+2*x**2+x**3
print('Derivative:',sp.diff(p,x))
q=1+2*x+x**2
print('Integral:',sp.integrate(q,(x,0,1)))

> **Closing note**
>
> A linear transformation is completely controlled by what it does to a basis. This principle leads directly to matrix representations in Section 6.3.

## Definitions and terminology

| Term | Meaning |
|---|---|
| Transformation | Function between vector spaces. |
| Domain | Set of allowable input vectors. |
| Codomain | Declared target vector space. |
| Image | Output $T(\mathbf v)$ of a vector. |
| Preimage | Input vector mapped to a specified output. |
| Range | Set of all images of vectors in the domain. |
| Linear transformation | Mapping preserving addition and scalar multiplication. |
| Zero transformation | Transformation mapping every vector to zero. |
| Differential operator | Linear mapping sending a function to its derivative. |

## Section 6.1 worksheet

1. For $T(x,y)=(2x-y,x+3y)$, find $T(4,-2)$.
2. Find the preimage of $(5,6)$ under the transformation in Problem 1.
3. Verify that $T(x,y)=(x-2y,3x+y)$ is linear.
4. Explain why $F(x,y)=(x+1,y)$ is not linear.
5. Suppose $T(\mathbf e_1)=(9,1)$, $T(\mathbf e_2)=(20,-3)$, and $T(\mathbf e_3)=(21,4)$. Find $T(1,-2,3)$.
6. Find the derivative of $3-2x+5x^2-x^3$ and evaluate $\int_0^1(1+x)\,dx$.

# Section 6.2 - Kernel and Range of a Linear Transformation

## Learning objectives

Students should be able to:

- find and interpret the kernel of a linear transformation;
- find a basis for the range;
- compute rank and nullity;
- apply the rank-nullity theorem;
- determine whether a transformation is one-to-one or onto;
- recognize isomorphic finite-dimensional vector spaces.

> **Kernel and range**
>
> For $T:V\to W$, the kernel is $\ker(T)=\{\mathbf v\in V:T(\mathbf v)=\mathbf 0\}$. The range is $\operatorname{range}(T)=\{T(\mathbf v):\mathbf v\in V\}$. Both are subspaces.

## Projection onto the $xy$-plane

Consider

$$
T(x,y,z)=(x,y,0).
$$

The kernel consists of vectors satisfying

$$
T(x,y,z)=(0,0,0),
$$

so $x=0$ and $y=0$, while $z$ is free. Therefore,

$$
\ker(T)=\{(0,0,z):z\in\mathbb R\}
=\operatorname{span}\{(0,0,1)\}.
$$

The range is the $xy$-plane:

$$
\operatorname{range}(T)
=\operatorname{span}\{(1,0,0),(0,1,0)\}.
$$

In [ ]:
# Visualize the kernel and range of the projection onto the xy-plane.
fig=plt.figure(figsize=(7,6))
ax=fig.add_subplot(111,projection='3d')
xx,yy=np.meshgrid(np.linspace(-3,3,10),np.linspace(-3,3,10))
zz=np.zeros_like(xx)
ax.plot_surface(xx,yy,zz,alpha=.25)
ax.plot([0,0],[0,0],[-3,3],linewidth=3,label='kernel: z-axis')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('Range plane and kernel line'); ax.legend(); plt.show()

## A rank-nullity example

Let $T:\mathbb R^4\to\mathbb R^3$ be defined by $T(\mathbf x)=A\mathbf x$, where

$$
A=
\begin{bmatrix}
1&0&2&1\\
0&1&-1&2\\
1&1&0&1
\end{bmatrix}.
$$

Row reduction gives

$$
\begin{aligned}
\begin{bmatrix}
1&0&2&1\\
0&1&-1&2\\
1&1&0&1
\end{bmatrix}
&\xrightarrow{R_3\leftarrow R_3-R_1}
\begin{bmatrix}
1&0&2&1\\
0&1&-1&2\\
0&1&-2&0
\end{bmatrix}\\
&\xrightarrow{R_3\leftarrow R_3-R_2}
\begin{bmatrix}
1&0&2&1\\
0&1&-1&2\\
0&0&-1&-2
\end{bmatrix}.
\end{aligned}
$$

There are three pivots, so

$$
\operatorname{rank}(T)=3.
$$

To find the kernel, solve

$$
\begin{aligned}
x_1+2x_3+x_4&=0,\\
x_2-x_3+2x_4&=0,\\
-x_3-2x_4&=0.
\end{aligned}
$$

Let $x_4=t$. Then

$$
x_3=-2t,
\qquad
x_2=-4t,
\qquad
x_1=3t.
$$

Thus,

$$
\ker(T)=\operatorname{span}\{(3,-4,-2,1)\},
$$

and

$$
\operatorname{nullity}(T)=1.
$$

The rank-nullity theorem checks:

$$
3+1=4=\dim(\mathbb R^4).
$$

In [ ]:
A=sp.Matrix([[1,0,2,1],[0,1,-1,2],[1,1,0,1]])
show_matrix(A,'A')
show_matrix(A.rref()[0],r'\operatorname{rref}(A)')
print('rank =',A.rank())
print('nullspace basis =',A.nullspace())

> **Rank-nullity theorem**
>
> If $T:V\to W$ is linear and $V$ is finite dimensional, then $\operatorname{rank}(T)+\operatorname{nullity}(T)=\dim(V)$.

## One-to-one and onto

A linear transformation is one-to-one exactly when

$$
\ker(T)=\{\mathbf 0\}.
$$

It is onto exactly when

$$
\operatorname{rank}(T)=\dim(W).
$$

For the $3\times4$ matrix above, the rank is $3$, equal to the dimension of the codomain $\mathbb R^3$. Therefore, $T$ is onto. Its nullity is $1$, so it is not one-to-one.

## Isomorphism example

Define

$$
C:P_2\to\mathbb R^3,
\qquad
C(a+bx+cx^2)=(a,b,c).
$$

This mapping is linear, one-to-one, and onto. Its inverse is

$$
C^{-1}(a,b,c)=a+bx+cx^2.
$$

Therefore, $P_2$ and $\mathbb R^3$ are isomorphic.

> **Closing note**
>
> The kernel measures information lost by a transformation, while the range measures the outputs that can actually be produced.

## Definitions and terminology

| Term | Meaning |
|---|---|
| Kernel | Inputs mapped to the zero vector. |
| Range | Set of all attainable outputs. |
| Nullity | Dimension of the kernel. |
| Rank | Dimension of the range. |
| Rank-nullity theorem | $\operatorname{rank}(T)+\operatorname{nullity}(T)=\dim(V)$. |
| One-to-one | Distinct inputs have distinct outputs. |
| Onto | Every vector in the codomain is an output. |
| Isomorphism | Linear transformation that is one-to-one and onto. |
| Isomorphic spaces | Vector spaces with the same linear structure. |

## Section 6.2 worksheet

1. Find the kernel and range of $T(x,y,z)=(x,y,0)$.
2. For
   $$
   A=\begin{bmatrix}1&0&2&1\\0&1&-1&2\\1&1&0&1\end{bmatrix},
   $$
   find a basis for $\ker(A)$ and the rank of $A$.
3. Verify rank-nullity for the matrix in Problem 2.
4. Determine whether the transformation in Problem 2 is one-to-one, onto, both, or neither.
5. Show that $C(a+bx+cx^2)=(a,b,c)$ is an isomorphism from $P_2$ to $\mathbb R^3$.
6. Explain why the kernel of every linear transformation contains the zero vector.

# Section 6.3 - Matrices for Linear Transformations

## Learning objectives

Students should be able to:

- construct the standard matrix of a linear transformation;
- use matrix multiplication to evaluate transformations;
- form matrices for compositions;
- find inverse transformations;
- construct matrices relative to nonstandard bases;
- represent operators between polynomial spaces.

> **Standard matrix theorem**
>
> If $T:\mathbb R^n\to\mathbb R^m$ is linear, then its standard matrix is $A=[T(\mathbf e_1)\ \cdots\ T(\mathbf e_n)]$, and $T(\mathbf x)=A\mathbf x$.

## Standard matrix from a formula

Let

$$
T(x,y,z)=(x+2y-z,\,3x-y+4z).
$$

Compute the images of the standard basis vectors:

$$
T(1,0,0)=(1,3),
$$

$$
T(0,1,0)=(2,-1),
$$

$$
T(0,0,1)=(-1,4).
$$

Therefore,

$$
A=
\begin{bmatrix}
1&2&-1\\
3&-1&4
\end{bmatrix}.
$$

For $\mathbf v=(2,-1,3)$,

$$
T(\mathbf v)=A\mathbf v
=
\begin{bmatrix}
1&2&-1\\
3&-1&4
\end{bmatrix}
\begin{bmatrix}
2\\-1\\3
\end{bmatrix}
=
\begin{bmatrix}
-3\\19
\end{bmatrix}.
$$

In [ ]:
A=sp.Matrix([[1,2,-1],[3,-1,4]])
v=sp.Matrix([2,-1,3])
show_matrix(A,'A'); show_vector(A*v,r'T(\mathbf v)')

> **Composition**
>
> If $T_1$ has standard matrix $A_1$ and $T_2$ has standard matrix $A_2$, then $T_2\circ T_1$ has standard matrix $A_2A_1$. The order matters.

## Shear followed by rotation

Let

$$
A_1=
\begin{bmatrix}
1&1\\0&1
\end{bmatrix}
$$

represent a horizontal shear, and let

$$
A_2=
\begin{bmatrix}
0&-1\\1&0
\end{bmatrix}
$$

represent a counterclockwise $90^\circ$ rotation.

The composition “shear first, then rotate” has matrix

$$
A_2A_1
=
\begin{bmatrix}
0&-1\\1&0
\end{bmatrix}
\begin{bmatrix}
1&1\\0&1
\end{bmatrix}
=
\begin{bmatrix}
0&-1\\1&1
\end{bmatrix}.
$$

The reverse composition has matrix

$$
A_1A_2
=
\begin{bmatrix}
1&-1\\1&0
\end{bmatrix},
$$

which is different.

In [ ]:
A1=sp.Matrix([[1,1],[0,1]])
A2=sp.Matrix([[0,-1],[1,0]])
show_matrix(A2*A1,r'A_2A_1')
show_matrix(A1*A2,r'A_1A_2')

## Inverse transformation

Let

$$
T(\mathbf x)=A\mathbf x,
\qquad
A=
\begin{bmatrix}
2&1\\1&1
\end{bmatrix}.
$$

Because

$$
\det(A)=2(1)-1(1)=1\neq0,
$$

$T$ is invertible. The inverse matrix is

$$
A^{-1}
=
\begin{bmatrix}
1&-1\\-1&2
\end{bmatrix}.
$$

Thus,

$$
T^{-1}(u,v)=(u-v,-u+2v).
$$

In [ ]:
A=sp.Matrix([[2,1],[1,1]])
show_matrix(A.inv(),r'A^{-1}')
print('Check A*A^{-1}:'); display(A*A.inv())

## A matrix relative to nonstandard bases

Let

$$
T(x,y)=(x+y,2x-y).
$$

Use the domain basis

$$
B=\{(1,1),(1,-1)\}
$$

and codomain basis

$$
C=\{(1,0),(1,1)\}.
$$

For the first basis vector,

$$
T(1,1)=(2,1)=1(1,0)+1(1,1),
$$

so

$$
[T(1,1)]_C=
\begin{bmatrix}1\\1\end{bmatrix}.
$$

For the second basis vector,

$$
T(1,-1)=(0,3)=-3(1,0)+3(1,1),
$$

so

$$
[T(1,-1)]_C=
\begin{bmatrix}-3\\3\end{bmatrix}.
$$

Therefore,

$$
[T]_{C\leftarrow B}
=
\begin{bmatrix}
1&-3\\
1&3
\end{bmatrix}.
$$

In [ ]:
M=sp.Matrix([[1,-3],[1,3]])
coords=sp.Matrix([2,-1])
show_matrix(M,r'[T]_{C\leftarrow B}')
show_vector(M*coords,r'[T(\mathbf v)]_C')
# Coordinate check: v=2(1,1)-(1,-1)=(1,3), T(v)=(4,-1).

## Differential operator matrix

Relative to the standard bases

$$
B=\{1,x,x^2,x^3\}
$$

for $P_3$ and

$$
C=\{1,x,x^2\}
$$

for $P_2$, differentiation has matrix

$$
[D]_{C\leftarrow B}
=
\begin{bmatrix}
0&1&0&0\\
0&0&2&0\\
0&0&0&3
\end{bmatrix}.
$$

Indeed, the columns are the coordinate vectors of

$$
D(1)=0,
\quad
D(x)=1,
\quad
D(x^2)=2x,
\quad
D(x^3)=3x^2.
$$

> **Closing note**
>
> The matrix of a transformation is a coordinate description. Changing bases changes the matrix but not the underlying transformation.

## Definitions and terminology

| Term | Meaning |
|---|---|
| Standard matrix | Matrix using standard bases in domain and codomain. |
| Composition | Applying one transformation after another. |
| Inverse transformation | Mapping that reverses an invertible transformation. |
| Transformation matrix | Matrix relative to chosen domain and codomain bases. |
| Coordinate vector | Coefficients of a vector relative to an ordered basis. |
| Differential-operator matrix | Matrix representing differentiation between polynomial spaces. |

## Section 6.3 worksheet

1. Find the standard matrix for $T(x,y,z)=(x+2y-z,3x-y+4z)$.
2. Use the matrix from Problem 1 to find $T(1,2,-1)$.
3. Let $A_1=\begin{bmatrix}1&1\\0&1\end{bmatrix}$ and $A_2=\begin{bmatrix}0&-1\\1&0\end{bmatrix}$. Find $A_2A_1$ and $A_1A_2$.
4. Find the inverse transformation for $T(\mathbf x)=\begin{bmatrix}2&1\\1&1\end{bmatrix}\mathbf x$.
5. Find the matrix of $T(x,y)=(x+y,2x-y)$ relative to $B=\{(1,1),(1,-1)\}$ and $C=\{(1,0),(1,1)\}$.
6. Write the matrix of differentiation $D:P_3\to P_2$ relative to the standard polynomial bases.

# Section 6.4 - Transition Matrices and Similarity

## Learning objectives

Students should be able to:

- construct transition matrices between ordered bases;
- convert coordinate vectors from one basis to another;
- relate matrices of the same transformation in different bases;
- test and use matrix similarity;
- recognize invariants preserved by similarity;
- exploit diagonal representations to compute matrix powers.

> **Transition matrix**
>
> If $B$ and $C$ are ordered bases, the transition matrix from $B$-coordinates to $C$-coordinates satisfies $[\mathbf x]_C=P_{C\leftarrow B}[\mathbf x]_B$. If the basis matrices are also called $B$ and $C$, then $P_{C\leftarrow B}=C^{-1}B$.

## Coordinate conversion

Let

$$
B=\{(1,1),(1,-1)\},
\qquad
C=\{(1,0),(1,1)\}.
$$

The basis matrices are

$$
B=
\begin{bmatrix}
1&1\\1&-1
\end{bmatrix},
\qquad
C=
\begin{bmatrix}
1&1\\0&1
\end{bmatrix}.
$$

Therefore,

$$
P_{C\leftarrow B}
=C^{-1}B
=
\begin{bmatrix}
1&-1\\0&1
\end{bmatrix}
\begin{bmatrix}
1&1\\1&-1
\end{bmatrix}
=
\begin{bmatrix}
0&2\\1&-1
\end{bmatrix}.
$$

For $\mathbf x=(5,1)$,

$$
[\mathbf x]_B=
\begin{bmatrix}3\\2\end{bmatrix}.
$$

Then

$$
[\mathbf x]_C
=P_{C\leftarrow B}[\mathbf x]_B
=
\begin{bmatrix}
0&2\\1&-1
\end{bmatrix}
\begin{bmatrix}3\\2\end{bmatrix}
=
\begin{bmatrix}4\\1\end{bmatrix}.
$$

In [ ]:
B=sp.Matrix([[1,1],[1,-1]])
C=sp.Matrix([[1,1],[0,1]])
P=C.inv()*B
show_matrix(P,r'P_{C\leftarrow B}')
show_vector(P*sp.Matrix([3,2]),r'[\mathbf x]_C')

> **Similarity relation**
>
> If $A$ is the matrix of $T$ in one basis and $A_B$ is the matrix in basis $B$, then $A_B=P^{-1}AP$. Matrices related this way are similar.

## A diagonal representation

Consider

$$
A=
\begin{bmatrix}
2&1\\1&2
\end{bmatrix}
$$

and the basis

$$
B=\{(1,1),(1,-1)\}.
$$

The transition matrix from $B$ to the standard basis is

$$
P=
\begin{bmatrix}
1&1\\1&-1
\end{bmatrix},
$$

and

$$
P^{-1}
=\frac12
\begin{bmatrix}
1&1\\1&-1
\end{bmatrix}.
$$

Then

$$
\begin{aligned}
A_B
&=P^{-1}AP\\
&=
\frac12
\begin{bmatrix}
1&1\\1&-1
\end{bmatrix}
\begin{bmatrix}
2&1\\1&2
\end{bmatrix}
\begin{bmatrix}
1&1\\1&-1
\end{bmatrix}\\
&=
\begin{bmatrix}
3&0\\0&1
\end{bmatrix}.
\end{aligned}
$$

Thus, the same transformation that mixes the standard coordinates simply scales one $B$-direction by $3$ and the other by $1$.

In [ ]:
A=sp.Matrix([[2,1],[1,2]])
P=sp.Matrix([[1,1],[1,-1]])
D=P.inv()*A*P
show_matrix(D,r'P^{-1}AP')
print('trace(A), trace(D):',A.trace(),D.trace())
print('det(A), det(D):',A.det(),D.det())

## Similarity invariants

Similar matrices have the same:

- determinant;
- trace;
- characteristic polynomial;
- eigenvalues;
- rank.

For the matrices above,

$$
\operatorname{tr}(A)=4=\operatorname{tr}(A_B),
$$

and

$$
\det(A)=3=\det(A_B).
$$

## Computing powers through similarity

Because

$$
A=PDP^{-1},
\qquad
D=
\begin{bmatrix}3&0\\0&1\end{bmatrix},
$$

we have

$$
A^n=PD^nP^{-1}.
$$

Since

$$
D^n=
\begin{bmatrix}3^n&0\\0&1\end{bmatrix},
$$

$$
A^n
=\frac12
\begin{bmatrix}
3^n+1&3^n-1\\
3^n-1&3^n+1
\end{bmatrix}.
$$

For example,

$$
A^5=
\begin{bmatrix}
122&121\\121&122
\end{bmatrix}.
$$

In [ ]:
A=sp.Matrix([[2,1],[1,2]])
show_matrix(A**5,r'A^5')

> **Closing note**
>
> Similarity separates an underlying linear transformation from the coordinate system used to describe it. A well-chosen basis can turn a complicated matrix into a simple one.

## Definitions and terminology

| Term | Meaning |
|---|---|
| Transition matrix | Converts coordinate vectors between ordered bases. |
| Basis matrix | Matrix whose columns are the basis vectors. |
| Similar matrices | Matrices related by $A_B=P^{-1}AP$. |
| Similarity invariant | Quantity unchanged by a basis change. |
| Diagonal representation | Matrix with only diagonal entries in a suitable basis. |
| Coordinate conversion | Rewriting the same vector in a different basis. |

## Section 6.4 worksheet

1. For $B=\{(1,1),(1,-1)\}$ and $C=\{(1,0),(1,1)\}$, find $P_{C\leftarrow B}$.
2. Convert $[\mathbf x]_B=(3,2)$ to $C$-coordinates.
3. For $A=\begin{bmatrix}2&1\\1&2\end{bmatrix}$ and $P=\begin{bmatrix}1&1\\1&-1\end{bmatrix}$, compute $P^{-1}AP$.
4. Verify that $A$ and $P^{-1}AP$ have the same trace and determinant.
5. Use the diagonal representation to compute $A^5$.
6. Explain why the order of vectors in a basis matters.

# Section 6.5 - Applications of Linear Transformations

## Learning objectives

Students should be able to:

- identify matrices for reflections, expansions, contractions, shears, and rotations;
- transform planar figures using matrix multiplication;
- compose geometric transformations;
- interpret determinant as area scaling;
- rotate three-dimensional objects about coordinate axes.

> **Common planar transformations**
>
> Reflections, scalings, shears, and rotations are linear transformations of $\mathbb R^2$. Their matrices describe how the standard basis vectors move.

## Standard $2\times2$ geometry matrices

Reflection in the $y$-axis:

$$
R_y=
\begin{bmatrix}-1&0\\0&1\end{bmatrix}.
$$

Reflection in the line $y=x$:

$$
R_{y=x}=
\begin{bmatrix}0&1\\1&0\end{bmatrix}.
$$

Horizontal scaling:

$$
S_x(k)=
\begin{bmatrix}k&0\\0&1\end{bmatrix}.
$$

Horizontal shear:

$$
H(k)=
\begin{bmatrix}1&k\\0&1\end{bmatrix}.
$$

Counterclockwise rotation by $\theta$:

$$
Q(\theta)=
\begin{bmatrix}
\cos\theta&-\sin\theta\\
\sin\theta&\cos\theta
\end{bmatrix}.
$$

## Badminton-court shear

Model a small rectangular court outline by

$$
(0,0),\ (6,0),\ (6,3),\ (0,3).
$$

Apply the horizontal shear

$$
H\left(\frac12\right)
=
\begin{bmatrix}
1&1/2\\0&1
\end{bmatrix}.
$$

A point $(x,y)$ becomes

$$
\left(x+\frac12y,y\right).
$$

Thus, the upper corners move right by $3/2$ units, while the lower corners remain fixed.

In [ ]:
court=np.array([[0,0],[6,0],[6,3],[0,3],[0,0]],dtype=float)
H=np.array([[1,.5],[0,1]])
plot_polygon_transform(court,H,'Fictional badminton-court shear')

## Rotation of a travel marker

A point $(2,0)$ rotated counterclockwise by $30^\circ$ becomes

$$
\begin{aligned}
Q(30^\circ)
\begin{bmatrix}2\\0\end{bmatrix}
&=
\begin{bmatrix}
\sqrt3/2&-1/2\\
1/2&\sqrt3/2
\end{bmatrix}
\begin{bmatrix}2\\0\end{bmatrix}\\
&=
\begin{bmatrix}\sqrt3\\1\end{bmatrix}.
\end{aligned}
$$

In [ ]:
pt=np.array([2.0,0.0])
rot=rotation2(np.pi/6)
print(rot@pt)
plot_vectors_2d([pt,rot@pt],['original','rotated 30 degrees'],'Rotation of a travel marker')

## Composition and order

Suppose a fictional Spurs pennant is first stretched horizontally by a factor of $2$ and then rotated $90^\circ$ counterclockwise.

The scaling matrix is

$$
S=
\begin{bmatrix}2&0\\0&1\end{bmatrix},
$$

and the rotation matrix is

$$
Q=
\begin{bmatrix}0&-1\\1&0\end{bmatrix}.
$$

The combined matrix is

$$
QS
=
\begin{bmatrix}0&-1\\2&0\end{bmatrix}.
$$

The reverse order would produce

$$
SQ
=
\begin{bmatrix}0&-2\\1&0\end{bmatrix},
$$

so the two geometric results are different.

> **Area scaling**
>
> For a planar linear transformation with matrix $A$, areas are multiplied by $|\det(A)|$. A zero determinant collapses every two-dimensional region to a line or point.

For the horizontal shear

$$
H(k)=\begin{bmatrix}1&k\\0&1\end{bmatrix},
$$

$$
\det(H(k))=1,
$$

so shear preserves area.

For

$$
S=\begin{bmatrix}2&0\\0&1/2\end{bmatrix},
$$

$$
\det(S)=1,
$$

so this stretch-and-compress transformation also preserves area.

## Three-dimensional rotation about the $z$-axis

A counterclockwise rotation about the $z$-axis has matrix

$$
R_z(\theta)=
\begin{bmatrix}
\cos\theta&-\sin\theta&0\\
\sin\theta&\cos\theta&0\\
0&0&1
\end{bmatrix}.
$$

At $\theta=90^\circ$,

$$
R_z(90^\circ)(1,2,3)=(-2,1,3).
$$

The $z$-coordinate is unchanged.

In [ ]:
# Rotate a fictional travel-luggage prism around the z-axis.
verts=np.array([[0,0,0],[2,0,0],[2,1,0],[0,1,0],[0,0,1.5],[2,0,1.5],[2,1,1.5],[0,1,1.5]],dtype=float)
edges=[(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]
R=rotation3_z(np.pi/4)
rotverts=(R@verts.T).T
fig=plt.figure(figsize=(8,6))
ax=fig.add_subplot(111,projection='3d')
for i,j in edges:
    ax.plot([verts[i,0],verts[j,0]],[verts[i,1],verts[j,1]],[verts[i,2],verts[j,2]],alpha=.55)
    ax.plot([rotverts[i,0],rotverts[j,0]],[rotverts[i,1],rotverts[j,1]],[rotverts[i,2],rotverts[j,2]])
ax.set_xlabel('x');ax.set_ylabel('y');ax.set_zlabel('z');ax.set_title('Original and 45-degree rotated luggage prism');plt.show()

## Interactive planar transformation

In [ ]:
shape=np.array([[0,0],[4,1],[0,2],[0,0]],dtype=float)
if WIDGETS_AVAILABLE:
    @interact(theta=FloatSlider(value=30,min=-180,max=180,step=5,description='rotation'),
              shear=FloatSlider(value=.25,min=-1.5,max=1.5,step=.05,description='shear'))
    def explore_transform(theta=30,shear=.25):
        R=rotation2(np.deg2rad(theta))
        H=np.array([[1,shear],[0,1]])
        A=R@H
        plot_polygon_transform(shape,A,f'Rotate {theta} degrees after shear {shear:.2f}')
else:
    print('ipywidgets is unavailable; the static transformation plots remain usable.')

> **Closing note**
>
> Geometric transformations make matrix multiplication visible: each matrix moves every point, edge, and region according to the same linear rule.

## Definitions and terminology

| Term | Meaning |
|---|---|
| Reflection | Mirror transformation across a line or axis. |
| Expansion | Scaling that increases distance in a direction. |
| Contraction | Scaling that decreases distance in a direction. |
| Shear | Transformation sliding parallel layers by proportional amounts. |
| Rotation | Distance- and angle-preserving turn about an axis or the origin. |
| Area scale factor | $|\det(A)|$ for a planar transformation. |
| Rotation matrix | Orthogonal matrix representing a rigid rotation. |
| Right-hand rule | Convention determining positive 3D rotation direction. |

## Section 6.5 worksheet

1. Identify the transformations represented by
   $$
   \begin{bmatrix}-1&0\\0&1\end{bmatrix},
   \qquad
   \begin{bmatrix}1&2\\0&1\end{bmatrix}.
   $$
2. Reflect the point $(3,-2)$ across the line $y=x$.
3. Apply the horizontal shear $H(1/2)$ to the triangle with vertices $(0,0)$, $(4,0)$, and $(0,2)$.
4. Rotate $(2,0)$ counterclockwise by $30^\circ$.
5. Find the matrix for a horizontal stretch by $2$ followed by a $90^\circ$ counterclockwise rotation.
6. Rotate $(1,2,3)$ counterclockwise by $90^\circ$ about the $z$-axis.

# Chapter 6 synthesis

A linear transformation can be studied from several mutually reinforcing viewpoints:

$$
\text{formula}
\longleftrightarrow
\text{matrix}
\longleftrightarrow
\text{kernel and range}
\longleftrightarrow
\text{geometry}.
$$

Changing a basis changes the matrix, but it does not change the underlying transformation. Similarity is the algebraic language for this coordinate change.

## Core connections

$$
\ker(T)=\text{information lost},
\qquad
\operatorname{range}(T)=\text{outputs attainable}.
$$

$$
[T]_{\text{new basis}}
=P^{-1}[T]_{\text{old basis}}P.
$$

$$
\text{composition of transformations}
\longleftrightarrow
\text{matrix multiplication in reverse application order}.
$$